In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import pytensor
import pytensor.tensor as pt
import matplotlib.pyplot as plt

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")


In [ ]:
from mmm_lab.data_generation.baseline import generate_baseline_geo_data
from mmm_lab.data_generation.marketing import add_marketing_effects

np.random.seed(42)
baseline_df = generate_baseline_geo_data(n_geos=40, n_weeks=104, start_date='2023-01-01')
df = add_marketing_effects(baseline_df, channels=['tv', 'paid_search'])

national_truth_tv = df['effect_tv'].sum() / df['spend_tv'].sum()
national_truth_ps = df['effect_paid_search'].sum() / df['spend_paid_search'].sum()

print(f"National ROAS — TV: {national_truth_tv:.3f}, PS: {national_truth_ps:.3f}")
print(f"Media share: {(df['effect_tv'] + df['effect_paid_search']).mean() / df['total_bookings'].mean():.1%}")


In [ ]:
df_sorted = df.sort_values(['geo', 'date'])
n_geos = df['geo'].nunique()
n_time = df['date'].nunique()

tv_spend_matrix   = df_sorted['spend_tv'].values.reshape(n_geos, n_time)
ps_spend_matrix   = df_sorted['spend_paid_search'].values.reshape(n_geos, n_time)
y_matrix          = df_sorted['total_bookings'].values.reshape(n_geos, n_time)
demand_matrix     = df_sorted['demand_good'].values.reshape(n_geos, n_time)

pop_weights = df_sorted.groupby('geo')['population'].first().values
pop_weights = pop_weights / pop_weights.sum()

print(f"Panel shape: {y_matrix.shape}  (n_geos × n_time)")
print(f"y range: {y_matrix.min():.0f} – {y_matrix.max():.0f}")
print(f"TV spend range: ${tv_spend_matrix.min():.0f} – ${tv_spend_matrix.max():.0f}")
print(f"Demand range: {demand_matrix.min():.0f} – {demand_matrix.max():.0f}")


In [ ]:
def geometric_adstock_panel(spend_matrix, alpha):
    """
    Geometric adstock via matrix multiplication.
    Builds adstock weight matrix M[t,s] = α^(t-s) for t>=s, then computes
    adstocked = spend @ M.T — a single vectorized operation.
    spend_matrix: (n_geos, n_time) numpy array.
    """
    T = spend_matrix.shape[1]

    # Lag matrix: M[t, s] = t - s (positive = past, negative = future)
    i_idx = pt.arange(T).reshape((-1, 1)).astype('float64')  # (T, 1)
    j_idx = pt.arange(T).reshape((1, -1)).astype('float64')  # (1, T)
    lag = i_idx - j_idx                                       # (T, T)

    # Weight matrix: α^lag where lag >= 0, else 0
    M = pt.where(lag >= 0, alpha ** lag, pt.zeros_like(lag))  # (T, T)

    # Apply to all geos simultaneously: (n_geos, T) @ (T, T) = (n_geos, T)
    spend_t = pt.as_tensor_variable(spend_matrix.astype(np.float64))  # (n_geos, T)
    return pt.dot(spend_t, M.T)




def hill_saturation(x, ec, slope):
    """Hill function: x^slope / (x^slope + ec^slope). Returns values in [0, 1]."""
    return (x ** slope) / (x ** slope + ec ** slope)


In [ ]:
with pm.Model() as mmm:

    # === Adstock decay (per channel) ===
    # Tight "cheating" priors — pure validation that the model is structurally correct
    alpha_tv = pm.Beta('alpha_tv', alpha=20, beta=20)    # peaks sharply at 0.5
    alpha_ps = pm.Beta('alpha_ps', alpha=15, beta=25)    # peaks at ~0.38
    K_tv = pm.LogNormal('K_tv', mu=np.log(5000), sigma=0.05)   # very tight at 5000
    K_ps = pm.LogNormal('K_ps', mu=np.log(2000), sigma=0.05)   # very tight at 2000

    # Slope: fixed at 1.0 in Meridian (we fix TV=1.0, PS=1.5 to match DGP)
    # slope_tv = 1.0  (hardcoded)
    # slope_ps = 1.5  (hardcoded)

    # === ROAS priors: break-even (median = 1.0) ===
    # ROAS: to match what we used in Meridian (break-even comparison):
    roas_tv = pm.LogNormal('roas_tv', mu=0.0, sigma=0.3)
    roas_ps = pm.LogNormal('roas_ps', mu=0.0, sigma=0.3)

    # === Saturation half-point K in dollar terms ===
    # K_tv = pm.LogNormal('K_tv', mu=np.log(5000), sigma=0.3)  # 95% CI: [2800, 8900]
    # K_ps = pm.LogNormal('K_ps', mu=np.log(2000), sigma=0.3)

    # === Geo-specific deviations (non-centered parameterization) ===
    # β_{m,g} = β_m + η_m * Z_{m,g}, Z ~ Normal(0,1)
    eta_tv = pm.HalfNormal('eta_tv', sigma=0.5)
    eta_ps = pm.HalfNormal('eta_ps', sigma=0.5)
    Z_tv   = pm.Normal('Z_tv', mu=0, sigma=1, shape=n_geos)
    Z_ps   = pm.Normal('Z_ps', mu=0, sigma=1, shape=n_geos)

    # === Hierarchical geo baselines (τ_g from paper eq. 12) ===
    tau_mu    = pm.Normal('tau_mu', mu=0, sigma=y_matrix.mean())
    tau_sigma = pm.HalfNormal('tau_sigma', sigma=y_matrix.std())
    tau_g     = pm.Normal('tau_g', mu=tau_mu, sigma=tau_sigma, shape=n_geos)

    # === Demand control (captures temporal baseline variation) ===
    gamma = pm.HalfNormal('gamma', sigma=2.0)

    # === Adstock + Hill transformations ===
    tv_adstocked = geometric_adstock_panel(tv_spend_matrix, alpha_tv)  # (n_geos, n_time)
    ps_adstocked = geometric_adstock_panel(ps_spend_matrix, alpha_ps)

    tv_sat = hill_saturation(tv_adstocked, K_tv, slope=1.0)   # (n_geos, n_time)
    ps_sat = hill_saturation(ps_adstocked, K_ps, slope=1.5)  # DGP uses 1.5; Meridian fixes at 1.0

    # === ROAS reparameterization (Google 2024 paper, eq. 11) ===
    # β_m = (ROAS_m × Σ spend - η_m × Σ Z[g]*hill[g,t]) / Σ hill[g,t]
    a_tv = roas_tv * tv_spend_matrix.sum()
    a_ps = roas_ps * ps_spend_matrix.sum()

    b_tv = eta_tv * (Z_tv[:, None] * tv_sat).sum()
    b_ps = eta_ps * (Z_ps[:, None] * ps_sat).sum()

    c_tv = tv_sat.sum()
    c_ps = ps_sat.sum()

    beta_tv = pm.Deterministic('beta_tv', (a_tv - b_tv) / c_tv)
    beta_ps = pm.Deterministic('beta_ps', (a_ps - b_ps) / c_ps)

    # === Geo-level effects (n_geos, n_time) ===
    tv_effect = (beta_tv + eta_tv * Z_tv[:, None]) * tv_sat
    ps_effect = (beta_ps + eta_ps * Z_ps[:, None]) * ps_sat

    # === Likelihood ===
    mu    = tau_g[:, None] + gamma * demand_matrix + tv_effect + ps_effect
    sigma = pm.HalfNormal('sigma', sigma=y_matrix.std())
    pm.Normal('y', mu=mu, sigma=sigma, observed=y_matrix)

print(mmm.free_RVs)


In [ ]:
with mmm:
    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.9,
        random_seed=42,
    )

print(az.summary(trace, var_names=['roas_tv', 'roas_ps', 'alpha_tv', 'alpha_ps', 
                                    'K_tv', 'K_ps'])[['mean', 'hdi_3%', 'hdi_97%', 'r_hat']])
